In [177]:
!pip install flask pyngrok opencv-python pillow numpy scikit-learn xgboost timm torch torchvision tensorflow


In [178]:
!pip install timm scikit-image torch torchvision tensorflow


In [179]:
!mkdir templates
!mkdir static

!mkdir static/uploads

mkdir: cannot create directory ‘templates’: File exists
mkdir: cannot create directory ‘static’: File exists
mkdir: cannot create directory ‘static/uploads’: File exists


In [180]:
from google.colab import drive    # connects drive to this Colab
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [181]:
%%writefile app.py

from flask import Flask, render_template, request
import os
import cv2
import numpy as np
import joblib
from PIL import Image

# ======== NEW IMPORTS FOR HYBRID FEATURES =========
import torch
import timm
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from skimage.feature import graycomatrix, graycoprops

# ======== LOAD MODELS =========
xgb = joblib.load("/content/drive/My Drive/finalModels/xgb_skin_cancer_model.pkl")
scaler = joblib.load("/content/drive/My Drive/finalModels/scaler.pkl")
pca = joblib.load("/content/drive/My Drive/finalModels/pca.pkl")
le = joblib.load("/content/drive/My Drive/finalModels/label_encoder.pkl")

IMG_SIZE = 192

app = Flask(__name__)
UPLOAD_FOLDER = "static/uploads"
app.config["UPLOAD_FOLDER"] = UPLOAD_FOLDER


# =====================================================
# LOAD FEATURE EXTRACTORS (SAME AS TRAINING)
# =====================================================

# ===== CNN EfficientNet =====
cnn_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    pooling='avg',
    input_shape=(192,192,3)
)

# ===== Swin Transformer =====
device = 'cuda' if torch.cuda.is_available() else 'cpu'

swin = timm.create_model(
    'swin_tiny_patch4_window7_224',
    pretrained=True,
    num_classes=0
).to(device)

swin.eval()


# =====================================================
# HANDCRAFTED FEATURES (FINAL VERSION)
# =====================================================
def handcrafted(img):
    gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_BGR2GRAY)


    glcm = graycomatrix(
        gray,
        [1],
        [0,np.pi/4,np.pi/2,3*np.pi/4],
        256,
        symmetric=True
    )

    contrast = graycoprops(glcm,'contrast').flatten()
    energy = graycoprops(glcm,'energy').flatten()
    homogeneity = graycoprops(glcm,'homogeneity').flatten()
    correlation = graycoprops(glcm,'correlation').flatten()

    return np.hstack([contrast,energy,homogeneity,correlation])


# =====================================================
# 🔥 REAL HYBRID FEATURE EXTRACTION
# =====================================================
def extract_features(img):

    # ---------- Preprocess ----------
    img = cv2.resize(img,(192,192))
    img = img / 255.0

    # ---------- CNN FEATURES ----------
    cnn_feat = cnn_model.predict(
        img[np.newaxis,...],
        verbose=0
    )

    # ---------- SWIN FEATURES ----------
    img_224 = cv2.resize(img,(224,224))

    tensor = torch.tensor(img_224)\
        .permute(2,0,1)\
        .unsqueeze(0)\
        .float()\
        .to(device)

    with torch.no_grad():
        swin_feat = swin(tensor).cpu().numpy()

    # ---------- HANDCRAFTED FEATURES ----------
    hc_feat = handcrafted(img).reshape(1,-1)

    # ---------- FEATURE FUSION ----------
    fused = np.concatenate(
        [cnn_feat, swin_feat, hc_feat],
        axis=1
    )

    return fused


# =====================================================
# PREDICTION FUNCTION
# =====================================================
def predict_image(img_path):

    img = cv2.imread(img_path)


    fused = extract_features(img)

    fused = scaler.transform(fused)
    fused = pca.transform(fused)

    pred = xgb.predict(fused)
    label = le.inverse_transform(pred)[0]
    print("Fused shape:", fused.shape)
    probs = xgb.predict_proba(fused)
    print("Probs:", probs)



    return label
def compute_scorecam(model, img_array, class_idx, layer_name):
    cam_model = Model(
        inputs=model.input,
        outputs=[
            model.get_layer(layer_name).output,
            model.output
        ]
    )

    conv_outputs, predictions = cam_model(img_array)
    conv_outputs = conv_outputs[0].numpy()  # (H, W, C)

    scorecam = np.zeros(conv_outputs.shape[:2], dtype=np.float32)

    for i in range(conv_outputs.shape[-1]):
        fmap = conv_outputs[:, :, i]
        fmap = np.maximum(fmap, 0)

        if np.max(fmap) == 0:
            continue

        fmap_norm = fmap / np.max(fmap)
        fmap_resized = cv2.resize(fmap_norm, (IMG_SIZE, IMG_SIZE))

        masked_img = img_array[0] * fmap_resized[..., np.newaxis]
        masked_img = np.expand_dims(masked_img, axis=0)

        score = model.predict(masked_img, verbose=0)[0][class_idx]
        scorecam += score * fmap

    scorecam = np.maximum(scorecam, 0)
    scorecam = scorecam / np.max(scorecam)

    return scorecam


# =====================================================
# ROUTES
# =====================================================
@app.route("/", methods=["GET","POST"])
def index():

    prediction = None
    img_path = None

    if request.method == "POST":

        file = request.files["file"]

        if file:
            save_path = os.path.join(
                app.config["UPLOAD_FOLDER"],
                file.filename
            )
            file.save(save_path)

            prediction = predict_image(save_path)
            img_path = save_path

    return render_template(
        "index.html",
        prediction=prediction,
        img_path=img_path
    )


if __name__ == "__main__":
    app.run(port=5000, host="0.0.0.0")


Overwriting app.py


In [182]:
%%writefile templates/index.html

<!DOCTYPE html>
<html>
<head>
    <title>FusionDerm - Hybrid Skin Cancer Classification</title>

    <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600&display=swap" rel="stylesheet">

    <link rel="stylesheet"
    href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.5.0/css/all.min.css">

    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>

<body>

<!-- NAVBAR -->
<nav class="navbar">
    <div class="logo">
        <i class="fa-solid fa-stethoscope"></i> FusionDerm
    </div>

    <div class="nav-links">
        <a href="#" onclick="showHome()">Home</a>
        <a href="#" onclick="showAbout()">About</a>
    </div>
</nav>


<!-- HOME SECTION -->
<section id="home-section">

<div class="header">
    <h1>Hybrid Skin Cancer Classification</h1>
    <p>AI Assisted Dermoscopic Lesion Analysis</p>
</div>

<div class="container">

<form method="POST" enctype="multipart/form-data">

    <div class="upload-title">
        <i class="fa-solid fa-image"></i> Upload Dermoscopic Image
    </div>

    <!-- ⭐ FLEX ROW (FIX BUTTON ALIGNMENT) -->
    <div class="upload-row">
        <input type="file" name="file" required>
        <button type="submit">
            <i class="fa-solid fa-microscope"></i> Predict
        </button>
    </div>

</form>

{% if img_path %}
    <img src="{{ img_path }}" class="preview">
{% endif %}

{% if prediction %}
<div class="result-box">
    <p class="result-title">
        <i class="fa-solid fa-file-medical"></i> Diagnosis Result
    </p>
    <h2 class="result-value">{{ prediction }}</h2>
</div>
{% endif %}

</div>
</section>



<!-- ABOUT SECTION -->
<section id="about-section" class="about-section" style="display:none;">

<h2>About FusionDerm</h2>

<p>
FusionDerm is a hybrid AI-based skin cancer classification system designed to assist
dermoscopic lesion analysis using advanced deep learning and machine learning techniques.
</p>

<ul>
<li>✔ EfficientNet extracts deep spatial image features</li>
<li>✔ Swin Transformer captures global contextual patterns</li>
<li>✔ Handcrafted texture features enhance lesion characterization</li>
<li>✔ Feature Fusion combines multiple representations</li>
<li>✔ PCA reduces dimensionality</li>
<li>✔ SMOTE balances class distribution</li>
<li>✔ XGBoost / LightGBM perform final classification</li>
</ul>

</section>


<script>
function showAbout(){
    document.getElementById("home-section").style.display="none";
    document.getElementById("about-section").style.display="block";
}
function showHome(){
    document.getElementById("home-section").style.display="block";
    document.getElementById("about-section").style.display="none";
}
</script>

</body>
</html>


Overwriting templates/index.html


In [183]:
%%writefile static/style.css

/* ============================= */
/* GLOBAL BODY */
/* ============================= */

body{
    margin:0;
    font-family:'Poppins',sans-serif;
    background:#6495ED;

    background-image:
        radial-gradient(rgba(255,255,255,0.08) 1px, transparent 1px);
    background-size:30px 30px;
    color:#0a2540;
}


/* ============================= */
/* NAVBAR */
/* ============================= */

.navbar{
    display:flex;
    justify-content:space-between;
    align-items:center;
    padding:18px 40px;
    color:white;
}

.logo{
    font-size:22px;
    font-weight:600;
}

.nav-links a{
    margin-left:25px;
    color:white;
    text-decoration:none;
    font-size:16px;
    transition:0.3s;
}

.nav-links a:hover{
    opacity:0.7;
}


/* ============================= */
/* HEADER */
/* ============================= */

.header{
    text-align:center;
    margin-top:40px;
    color:white;
}

.header h1{
    font-size:42px;
    margin-bottom:10px;
    font-weight:600;
}

.header p{
    font-size:18px;
    opacity:0.9;
}


/* ============================= */
/* GLASS CARD */
/* ============================= */

.container{
    width:40%;
    margin:40px auto;
    padding:40px;

    background:rgba(255,255,255,0.18);
    backdrop-filter:blur(14px);
    border-radius:20px;

    box-shadow:0 10px 30px rgba(0,0,0,0.15);
}


/* ============================= */
/* UPLOAD SECTION */
/* ============================= */

.upload-title{
    font-size:18px;
    margin-bottom:15px;
}

/* ⭐ FIX ALIGNMENT */
.upload-row{
    display:flex;
    justify-content:center;
    align-items:center;
    gap:15px;
    flex-wrap:nowrap;
}

input[type="file"]{
    margin:0;
}


/* ============================= */
/* BUTTON */
/* ============================= */

button{
    padding:12px 26px;
    background:#3b6fc4;
    color:white;
    border:none;
    border-radius:8px;
    font-size:16px;
    cursor:pointer;
    transition:0.3s;
}

button:hover{
    background:#2f5ca5;
    transform:translateY(-2px);
    box-shadow:0 5px 12px rgba(0,0,0,0.2);
}


/* ============================= */
/* PREVIEW IMAGE */
/* ============================= */

.preview{
    display:block;
    width:260px;
    margin:25px auto;
    border-radius:12px;
    box-shadow:0 4px 15px rgba(0,0,0,0.2);
}


/* ============================= */
/* RESULT BOX */
/* ============================= */

.result-box{
    margin-top:25px;
    padding:25px;
    border-radius:14px;

    background:rgba(255,255,255,0.35);
    box-shadow:0 0 15px rgba(0,0,0,0.15);

    text-align:center;
}

.result-title{
    font-size:14px;
    opacity:0.8;
}

.result-value{
    font-size:32px;
    font-weight:700;
    margin-top:8px;
    letter-spacing:1px;
}


/* ============================= */
/* ABOUT SECTION */
/* ============================= */

.about-section{
    width:60%;
    margin:60px auto;
    padding:40px;

    background:rgba(255,255,255,0.2);
    backdrop-filter:blur(10px);

    border-radius:18px;
    color:#0a2540;
}

.about-section h2{
    text-align:center;
    margin-bottom:20px;
}

html{
    scroll-behavior:smooth;
}


Overwriting static/style.css


In [206]:
!pkill -f flask || echo "No flask running"
!pkill -f ngrok || echo "No ngrok running"



^C
^C


In [207]:
# 🔎 List processes using port 8000
!lsof -i :5000



COMMAND   PID USER   FD   TYPE  DEVICE SIZE/OFF NODE NAME
python3 53760 root    5u  IPv4 1512927      0t0  TCP *:5000 (LISTEN)


In [208]:
# ❌ Kill process by PID (replace 12345 with actual PID from previous cell)
!kill -9 53760


In [209]:
# ============================
# ✅ Run Flask in background
# ============================
!nohup python app.py > flask.log 2>&1 &


In [216]:
!tail -n 50 flask.log

2026-02-11 07:58:36.173615: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-11 07:58:36.178837: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-11 07:58:36.192706: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770796716.216672   54168 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770796716.223733   54168 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770796716.241226   54168 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [217]:
# ============================
# ✅ Start ngrok tunnel
# ============================
from pyngrok import ngrok, conf

conf.get_default().auth_token = "39VZsUg3uHwUZ2kezN5CAzoXjEs_73437ghMuVKE88G1Uwrxr"   # 🔑 put your token here
public_url = ngrok.connect(5000)
print("🌍 Public URL:", public_url)

🌍 Public URL: NgrokTunnel: "https://gleamless-unartful-dylan.ngrok-free.dev" -> "http://localhost:5000"
